# Car Traffic Warning - train the detector

This teaches a model to find **traffic signals** and **speed limit boards**.

**How to use:** in the menu click **Runtime > Run all**. If a warning appears, click **Run anyway**. Keep this tab open until a file named **ctw_model.zip** downloads (30 to 60 minutes).

Training pictures: *Traffic Signs Detection* by Parisa Karimi Darabi (Kaggle), licence CC BY 4.0. Model: Ultralytics YOLO11 (AGPL-3.0).

In [ ]:
# STEP 1 - install the tools and check the GPU
!pip -q install ultralytics kagglehub
import torch
if torch.cuda.is_available():
    print("GPU OK:", torch.cuda.get_device_name(0))
else:
    print("NO GPU. In the menu choose: Runtime > Change runtime type > T4 GPU > Save, then run all again.")


In [ ]:
# STEP 2 - download the public training pictures (about 105 MB, licence CC BY 4.0)
import kagglehub, os, glob, yaml

try:
    root = kagglehub.dataset_download("pkdarabi/cardetection")
except Exception as e:
    raise SystemExit("DOWNLOAD FAILED. Copy this whole message and send it in the chat: " + repr(e))
print("Downloaded to:", root)

def find_split(name):
    hits = [d for d in glob.glob(os.path.join(root, "**", name), recursive=True)
            if os.path.isdir(os.path.join(d, "images")) and os.path.isdir(os.path.join(d, "labels"))]
    assert hits, "Could not find the '" + name + "' folder in the download"
    return sorted(hits, key=len)[0]

train_dir, val_dir = find_split("train"), find_split("valid")

NAMES = ["Green Light", "Red Light", "Speed Limit 10", "Speed Limit 100", "Speed Limit 110", "Speed Limit 120",
         "Speed Limit 20", "Speed Limit 30", "Speed Limit 40", "Speed Limit 50", "Speed Limit 60", "Speed Limit 70",
         "Speed Limit 80", "Speed Limit 90", "Stop"]
# use the class list shipped with the pictures when there is one
for y in glob.glob(os.path.join(root, "**", "data.yaml"), recursive=True):
    try:
        n = yaml.safe_load(open(y)).get("names")
        if isinstance(n, dict): n = [n[k] for k in sorted(n)]
        if n: NAMES = list(n); break
    except Exception as e:
        print("skipped", y, e)

yaml.safe_dump({"train": os.path.join(train_dir, "images"), "val": os.path.join(val_dir, "images"),
                "nc": len(NAMES), "names": NAMES}, open("/content/ctw.yaml", "w"))
print("Training pictures:", len(os.listdir(os.path.join(train_dir, "images"))))
print("Checking pictures:", len(os.listdir(os.path.join(val_dir, "images"))))
print("Classes:", NAMES)


In [ ]:
# STEP 3 - teach the model (about 30 to 60 minutes; keep this tab open)
from ultralytics import YOLO

model = YOLO("yolo11n.pt")
model.train(data="/content/ctw.yaml", epochs=50, imgsz=640, batch=32, patience=12, workers=2,
            project="/content/runs", name="ctw", exist_ok=True)
best_path = str(model.trainer.best)
print("Best model saved at:", best_path)


In [ ]:
# STEP 4 - pack the trained model and download it to your computer
import json, shutil, zipfile, os
from ultralytics import YOLO

best = YOLO(best_path)
onnx_path = best.export(format="onnx", imgsz=640, opset=12, simplify=False)
os.makedirs("/content/out", exist_ok=True)
shutil.copy(onnx_path, "/content/out/ctw_model.onnx")
shutil.copy(best_path, "/content/out/ctw_model.pt")
json.dump([best.names[i] for i in sorted(best.names)], open("/content/out/ctw_labels.json", "w"))
res = os.path.join(os.path.dirname(os.path.dirname(best_path)), "results.csv")
if os.path.exists(res): shutil.copy(res, "/content/out/results.csv")
with zipfile.ZipFile("/content/ctw_model.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for f in os.listdir("/content/out"): z.write(os.path.join("/content/out", f), f)
print("Finished. File size (MB):", round(os.path.getsize("/content/ctw_model.zip") / 1e6, 1))
try:
    from google.colab import files
    files.download("/content/ctw_model.zip")
except Exception as e:
    print("Automatic download did not start:", e)
    print("Open the folder icon on the left, right-click ctw_model.zip and choose Download.")


**Done.** Send the downloaded **ctw_model.zip** file in the chat.